# **TAHAP 2** *Case Representation*

In [20]:
import os
import re
import pandas as pd

# Load Data Clean

In [21]:
clean_dir = "/content/data/processed/clean_text"

# Normalisasi

In [22]:
def normalize(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

# Case Representation

In [23]:
rows = []

for file in valid:

    if not file.endswith(".txt"):
        continue

    with open(
        os.path.join(clean_dir, file),
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:
        text = f.read()

    text = normalize(text)

    # METADATA

    no_perkara = ""
    tanggal = ""
    terdakwa = ""

    m_no = re.search(
        r'putusan\s*nomor\s*[:\-]?\s*([0-9/\.a-z]+)',
        text
    )

    if m_no:
        no_perkara = m_no.group(1)

    m_tgl = re.search(
        r'tanggal\s*[:\-]?\s*([^,\.]+)',
        text
    )

    if m_tgl:
        tanggal = m_tgl.group(1)

    m_td = re.search(
        r'nama lengkap\s*[:\-]?\s*(.*?)(tempat lahir|umur|jenis kelamin|kebangsaan)',
        text
    )

    if m_td:
        terdakwa = m_td.group(1).strip()

    # FAKTA

    m_fakta = re.search(
        r'(menimbang.*?)(amar putusan|putusan|$)',
        text
    )

    if m_fakta:
        ringkasan_fakta = (
            m_fakta.group(1)[:1500]
        )
    else:
        ringkasan_fakta = (
            " ".join(
                text.split()[:300]
            )
        )

    # PASAL

    pasals = re.findall(
        r'pasal\s*[0-9a-z\s]+',
        text
    )

    pasal = ", ".join(
        sorted(
            set(pasals)
        )
    )

    # AMAR

    m_amar = re.search(
        r'amar\s*putusan(.*?)(menimbang|demikian|putusan|$)',
        text
    )

    if m_amar:
        amar_putusan = (
            m_amar.group(1)
            .strip()[:800]
        )
    else:
        amar_putusan = ""

    argumen_hukum = (
        pasal
        + " "
        + amar_putusan
    ).strip()

    word_count = len(
        text.split()
    )

    char_count = len(
        text
    )

    # LABEL

    jenis_perkara = "lain"

    if (
        "pemerasan" in text
        and
        "pengancaman" in text
    ):
        jenis_perkara = (
            "pemerasan_dan_pengancaman"
        )

    elif "pemerasan" in text:
        jenis_perkara = (
            "pemerasan"
        )

    elif "pengancaman" in text:
        jenis_perkara = (
            "pengancaman"
        )

    rows.append({

        "case_id":
        file.replace(
            ".txt",
            ""
        ),

        "no_perkara":
        no_perkara,

        "tanggal":
        tanggal,

        "jenis_perkara":
        jenis_perkara,

        "terdakwa":
        terdakwa,

        "pasal":
        pasal,

        "ringkasan_fakta":
        ringkasan_fakta,

        "argumen_hukum":
        argumen_hukum,

        "amar_putusan":
        amar_putusan,

        "word_count":
        word_count,

        "char_count":
        char_count,

        "text_full":
        text
    })

print(
    "TOTAL CASE:",
    len(rows)
)

TOTAL CASE: 34


In [24]:
df = pd.DataFrame(rows)

output_path = "/content/data/processed/cases.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("SAVED:", output_path)

print(
    "SHAPE:",
    df.shape
)

print(
    df[
        [
            "case_id",
            "jenis_perkara",
            "pasal"
        ]
    ].head()
)

print(
    df["jenis_perkara"]
    .value_counts()
)

SAVED: /content/data/processed/cases.csv
SHAPE: (34, 12)
    case_id jenis_perkara                                              pasal
0  case_026   pengancaman                                    pasal 335 ayat 
1  case_015   pengancaman     pasal 170 ayat , pasal 44 kuhp, pasal 55 ayat 
2  case_023   pengancaman                                    pasal 335 ayat 
3  case_009   pengancaman                                    pasal 335 ayat 
4  case_001   pengancaman  pasal 3 ayat , pasal 335 ayat , pasal 448 ayat...
jenis_perkara
pengancaman                  15
pemerasan_dan_pengancaman    10
pemerasan                     8
lain                          1
Name: count, dtype: int64
